 ## Введение
 Несмотря на название, логистическая регрессия не используется для задач регрессии, а применяется для предсказания категориальных классов.

 Модель не просто выдает класс, а сначала оценивает вероятность того, что объект принадлежит определенному классу, после чего на основе этой вероятности принимается итоговое решение.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.preprocessing import StandardScaler

model = LogisticRegression()
data = load_breast_cancer()
X, y = data.data, data.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## Сравнение с линейной регрессией

### Линейная регрессия

$$\hat{y} = \mathbf{w}^T\mathbf{x} = w_0 + w_1x_1 + w_2x_2 + \ldots + w_nx_n$$

$\mathbf{w}^T\mathbf{x}$ — это **скалярное произведение** вектора весов и вектора признаков:

$$\mathbf{w}^T\mathbf{x} = \begin{bmatrix} w_0 & w_1 & \cdots & w_n \end{bmatrix} \begin{bmatrix} x_0 \\ x_1 \\ \vdots \\ x_n \end{bmatrix} = w_0x_0 + w_1x_1 + \cdots + w_nx_n$$

- Предсказывает непрерывное значение
- Выход не ограничен: $\hat{y} \in (-\infty, +\infty)$ → нельзя интерпретировать как вероятность
- Подходит для задач регрессии (цена, температура, etc.)

### Логистическая регрессия
 $$\hat{p} = \sigma(\mathbf{w}^T\mathbf{x})$$


- Пропускает линейную комбинацию через сигмоиду
- Выход всегда в $[0, 1]$ → интерпретируется как вероятность класса
- Решение по порогу: $\hat{y} = 1$ если $\hat{p} \geq 0.5$
- Подходит для задач классификации

### Ключевое отличие

Несмотря на слово «регрессия» в названии — логистическая регрессия решает задачу **классификации**. Линейная часть $\mathbf{w}^T\mathbf{x}$ одинакова в обеих моделях, разница только в том, что с ней делают дальше.

In [ ]:
np.random.seed(42)
x_data = np.random.uniform(-8, 8, 40)
y_data = (x_data > 0).astype(float)

x = np.linspace(-10, 10, 300)
sigmoid = 1 / (1 + np.exp(-x))

from numpy.polynomial import polynomial as P
coeffs = np.polyfit(x_data, y_data, 1)
linear = np.polyval(coeffs, x)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))


threshold_linear = (0.5 - coeffs[1]) / coeffs[0]

axes[0].scatter(x_data[y_data == 0], y_data[y_data == 0] + np.random.normal(0, 0.02, sum(y_data==0)),
                color='steelblue', s=40, zorder=5, label='Класс 0')
axes[0].scatter(x_data[y_data == 1], y_data[y_data == 1] + np.random.normal(0, 0.02, sum(y_data==1)),
                color='tomato', s=40, zorder=5, label='Класс 1')
axes[0].plot(x, linear, color='green', linewidth=2, label='Линейная регрессия')
axes[0].axhline(0.5, color='red', linestyle='--', linewidth=1.5, label='Порог = 0.5')
axes[0].axvline(threshold_linear, color='red', linestyle=':', linewidth=1.2)
axes[0].fill_between(x, 0.5, linear, where=(linear >= 0.5), alpha=0.10, color='tomato', label='→ Класс 1')
axes[0].fill_between(x, 0.5, linear, where=(linear < 0.5),  alpha=0.10, color='steelblue', label='→ Класс 0')
axes[0].axhspan(-0.5, 0, alpha=0.06, color='tomato')
axes[0].axhspan(1.0, 1.5, alpha=0.06, color='tomato')
axes[0].set_ylim(-0.4, 1.4)
axes[0].set_xlim(-10, 10)
axes[0].set_xlabel('x'); axes[0].set_ylabel('Предсказание')
axes[0].set_title('Линейная регрессия\n(задача бинарной классификации)', fontsize=11)
axes[0].legend(fontsize=9, loc='upper left'); axes[0].grid(alpha=0.3)

axes[1].scatter(x_data[y_data == 0], y_data[y_data == 0] + np.random.normal(0, 0.02, sum(y_data==0)),
                color='steelblue', s=40, zorder=5, label='Класс 0')
axes[1].scatter(x_data[y_data == 1], y_data[y_data == 1] + np.random.normal(0, 0.02, sum(y_data==1)),
                color='tomato', s=40, zorder=5, label='Класс 1')
axes[1].plot(x, sigmoid, color='green', linewidth=2.5, label='Логистическая регрессия')
axes[1].axhline(0.5, color='red', linestyle='--', linewidth=1.5, label='Порог = 0.5')
axes[1].axvline(0, color='red', linestyle=':', linewidth=1.2)
axes[1].fill_between(x, sigmoid, 0.5, where=(sigmoid >= 0.5), alpha=0.12, color='tomato', label='→ Класс 1')
axes[1].fill_between(x, sigmoid, 0.5, where=(sigmoid <  0.5), alpha=0.12, color='steelblue', label='→ Класс 0')
axes[1].annotate('σ(0) = 0.5', xy=(0, 0.5), xytext=(1.5, 0.42),
                 arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)
axes[1].annotate('σ(+∞) → 1', xy=(8, 0.9997), xytext=(4.5, 0.82),
                 arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)
axes[1].annotate('σ(−∞) → 0', xy=(-8, 0.0003), xytext=(-9.5, 0.15),
                 arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)
axes[1].set_ylim(-0.05, 1.10)
axes[1].set_xlim(-10, 10)
axes[1].set_xlabel('z = wᵀx'); axes[1].set_ylabel('P(y=1 | x)')
axes[1].set_title('Логистическая регрессия\nВыход всегда в [0, 1]', fontsize=11)
axes[1].legend(fontsize=9, loc='upper left'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## **Параметры Logistic Regression**

In [ ]:
import inspect
sig = inspect.signature(LogisticRegression)
print("LogisticRegression(")
for name, param in sig.parameters.items():
    print(f"    {name} = {param.default!r}")
print(")")


## 1. `penalty` — тип регуляризации

Регуляризация добавляет штраф за большие веса в функцию потерь, чтобы модель не переобучалась. Параметр `penalty` определяет тип этого штрафа:

- `'l2'` (default) — уменьшает все веса равномерно
- `'l1'` — обнуляет незначимые веса, работает как отбор признаков
- `'elasticnet'` — комбинация L1 и L2
- `None` — без штрафа, риск переобучения

In [ ]:
import warnings

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

penalties = ['l2', 'l1', 'elasticnet', None]
solvers   = ['lbfgs', 'saga', 'saga', 'lbfgs']

for ax, penalty, solver in zip(axes, penalties, solvers):
    kwargs = dict(penalty=penalty, C=1.0, solver=solver, max_iter=2000)
    if penalty == 'elasticnet':
        kwargs['l1_ratio'] = 0.5

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        model = LogisticRegression(**kwargs)
        model.fit(X_scaled, y)

    coefs = model.coef_[0]
    n_zero = np.sum(np.abs(coefs) < 1e-4)
    colors = ['tomato' if abs(c) < 1e-4 else 'steelblue' for c in coefs]

    ax.bar(range(len(coefs)), coefs, color=colors, alpha=0.8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'penalty={penalty!r}\nнулевых: {n_zero}/30', fontsize=10)
    ax.set_xlabel('Признак')
    ax.grid(alpha=0.3)

axes[0].set_ylabel('Значение коэффициента')
plt.suptitle('Влияние penalty на коэффициенты модели (C=1.0)', fontsize=13)
plt.tight_layout()
plt.show()

## 2. `C` — сила регуляризации

`C` — это **обратная** сила регуляризации: чем меньше `C`, тем сильнее штраф и проще модель. Чем больше `C` — тем слабее регуляризация, модель сложнее и может переобучиться.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

C_values = [0.001, 0.01, 0.1, 1.0, 100.0]

for ax, C in zip(axes, C_values):
    model = LogisticRegression(C=C, max_iter=2000)
    model.fit(X_scaled, y)
    coefs = model.coef_[0]

    ax.bar(range(len(coefs)), coefs, color='steelblue', alpha=0.8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'C={C}\nacc={model.score(X_scaled, y):.3f}', fontsize=10)
    ax.set_xlabel('Признак')
    ax.grid(alpha=0.3)

axes[0].set_ylabel('Значение коэффициента')
plt.suptitle('Влияние C на коэффициенты модели (penalty=l2)', fontsize=13)
plt.tight_layout()
plt.show()

### 3. `solver` — алгоритм оптимизации

Параметр `solver` определяет метод, которым модель минимизирует функцию потерь. Разные солверы подходят для разных задач:

- `'lbfgs'` (default) — хорошо работает на большинстве задач, поддерживает только `l2` и `None`
- `'liblinear'` — подходит для малых датасетов, поддерживает `l1` и `l2`
- `'saga'` — работает на больших датасетах, поддерживает все типы `penalty`
- `'sag'` — похож на `saga`, но только `l2` и `None`
- `'newton-cg'` — только `l2` и `None`

Главное ограничение: **не все солверы совместимы со всеми типами `penalty`**.

In [ ]:
import time

solvers = ['lbfgs', 'liblinear', 'saga', 'sag', 'newton-cg']
times, accs, iters = [], [], []

for solver in solvers:
    t0 = time.time()
    model = LogisticRegression(solver=solver, max_iter=1000)
    model.fit(X_scaled, y)
    times.append(time.time() - t0)
    accs.append(model.score(X_scaled, y))
    iters.append(model.n_iter_[0])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(solvers, accs, color='steelblue', alpha=0.8)
axes[0].set_ylim(min(accs) - 0.01, 1.0)
axes[0].set_title('Точность', fontsize=12)
axes[0].set_ylabel('Accuracy')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(solvers, iters, color='green', alpha=0.8)
axes[1].set_title('Число итераций до сходимости', fontsize=12)
axes[1].set_ylabel('Итераций')
axes[1].grid(axis='y', alpha=0.3)

axes[2].bar(solvers, times, color='tomato', alpha=0.8)
axes[2].set_title('Время обучения', fontsize=12)
axes[2].set_ylabel('Время (сек)')
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('Сравнение солверов (penalty=l2)', fontsize=13)
plt.tight_layout()
plt.show()

## 4. `max_iter` — максимальное число итераций

Алгоритм обучается итерационно — на каждом шаге обновляет веса. `max_iter` задаёт максимальное число таких шагов.

- Если модель сошлась раньше — останавливается досрочно
- Если не сошлась за `max_iter` шагов — sklearn выдаёт `ConvergenceWarning` - это предупреждение от sklearn, которое означает что модель **не успела сойтись** за отведённое число итераций.
- По умолчанию: `max_iter=100`

Если видите предупреждение `ConvergenceWarning` — увеличьте `max_iter` или масштабируйте данные через `StandardScaler`.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

max_iters = [10, 20, 50, 100, 200, 500, 1000]
real_iters, converged = [], []

for n in max_iters:
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter('always')
        model = LogisticRegression(max_iter=n, solver='lbfgs')
        model.fit(X_scaled, y)
        did_converge = not any(issubclass(x.category, ConvergenceWarning) for x in w)
    real_iters.append(model.n_iter_[0])
    converged.append(did_converge)

colors = ['steelblue' if c else 'tomato' for c in converged]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar([str(n) for n in max_iters], max_iters, color='lightgray', alpha=0.6, label='max_iter (лимит)')
axes[0].bar([str(n) for n in max_iters], real_iters, color=colors, alpha=0.9, label='Реально использовано')
axes[0].set_xlabel('max_iter'); axes[0].set_ylabel('Итераций')
axes[0].set_title('Лимит vs реально использованные итерации', fontsize=11)
from matplotlib.patches import Patch
axes[0].legend(handles=[
    Patch(facecolor='lightgray', label='Лимит (max_iter)'),
    Patch(facecolor='steelblue', label='Сошлось'),
    Patch(facecolor='tomato', label='Не сошлось (достигнут лимит)'),
])
axes[0].grid(axis='y', alpha=0.3)

axes[1].plot([str(n) for n in max_iters], real_iters, 'o-', color='steelblue', linewidth=2)
for i, (n, r, c) in enumerate(zip(max_iters, real_iters, converged)):
    color = 'steelblue' if c else 'tomato'
    axes[1].scatter(str(n), r, color=color, s=80, zorder=5)
axes[1].axhline(real_iters[-1], color='gray', linestyle='--', alpha=0.6, label=f'Сходимость на {real_iters[-1]} итерации')
axes[1].set_xlabel('max_iter'); axes[1].set_ylabel('Итераций до остановки')
axes[1].set_title('Когда модель остановилась', fontsize=11)
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Влияние max_iter на сходимость', fontsize=13)
plt.tight_layout()
plt.show()

## 5. `class_weight` — веса классов

При несбалансированных данных (например, 95% класс 0 и 5% класс 1) модель склонна просто игнорировать меньший класс — так проще получить высокую точность.

`class_weight` позволяет это исправить, назначив меньшему классу больший вес при обучении:

- `None` (default) — все классы равнозначны
- `'balanced'` — веса автоматически обратно пропорциональны частоте класса
- `{0: 1, 1: 10}` — задать веса вручную

Главный индикатор проблемы — не `accuracy`, а `recall` по меньшему классу.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split

X_imb, y_imb = make_classification(
    n_samples=1000, n_features=2, n_redundant=0,
    weights=[0.95, 0.05], random_state=42
)
X_imb = StandardScaler().fit_transform(X_imb)
Xt, Xv, yt, yv = train_test_split(X_imb, y_imb, test_size=0.3, random_state=42)

print(f"Класс 0: {sum(yt==0)},  Класс 1: {sum(yt==1)}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, cw, title in zip(axes,
    [None, 'balanced', {0: 1, 1: 10}],
    ['None', "'balanced'", '{0:1, 1:10}']):

    model = LogisticRegression(class_weight=cw, max_iter=1000)
    model.fit(Xt, yt)
    preds = model.predict(Xv)

    cm = confusion_matrix(yv, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Класс 0', 'Класс 1'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')

    acc = accuracy_score(yv, preds)
    recall_1 = cm[1, 1] / cm[1].sum() if cm[1].sum() > 0 else 0
    ax.set_title(f'class_weight={title}\nAccuracy={acc:.3f},  Recall(1)={recall_1:.3f}', fontsize=10)

plt.suptitle('Влияние class_weight на несбалансированных данных', fontsize=13)
plt.tight_layout()
plt.show()

## 6. `tol` — критерий остановки

`tol` задаёт точность, при которой алгоритм считает что модель **сошлась** и останавливает обучение досрочно — до достижения `max_iter`.

На каждой итерации алгоритм проверяет: если изменение весов стало меньше `tol` — обучение останавливается.

- **Меньший `tol`** → модель обучается дольше, но точнее
- **Больший `tol`** → модель останавливается раньше, но может не досчитаться

По умолчанию: `tol=1e-4`

In [ ]:
tol_values = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
tol_iters, tol_accs = [], []

for tol in tol_values:
    model = LogisticRegression(tol=tol, max_iter=5000, solver='lbfgs')
    model.fit(X_scaled, y)
    tol_iters.append(model.n_iter_[0])
    tol_accs.append(model.score(X_scaled, y))

tol_labels = [f'1e-{i+1}' for i in range(len(tol_values))]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(tol_labels, tol_iters, color='steelblue', alpha=0.8)
axes[0].set_title('Число итераций до сходимости', fontsize=12)
axes[0].set_xlabel('tol'); axes[0].set_ylabel('Итераций')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(tol_labels, tol_accs, color='tomato', alpha=0.8)
axes[1].set_ylim(min(tol_accs) - 0.005, 1.0)
axes[1].set_title('Точность модели', fontsize=12)
axes[1].set_xlabel('tol'); axes[1].set_ylabel('Accuracy')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Влияние tol на сходимость и точность', fontsize=13)
plt.tight_layout()
plt.show()

## 7. `multi_class` — стратегия многоклассовой классификации (устарел)

По умолчанию логистическая регрессия решает задачу **бинарной** классификации. Для задач с тремя и более классами используется параметр `multi_class`:

- `'ovr'` (One-vs-Rest) — обучается K отдельных бинарных классификаторов, каждый отделяет один класс от всех остальных
- `'multinomial'` — одна модель на все классы сразу, использует функцию **Softmax** вместо сигмоиды
- `'auto'` (default) — выбирает автоматически: `multinomial` для `lbfgs` и `saga`, `ovr` для остальных

$$\text{Softmax}: P(y=k \mid x) = \frac{e^{w_k^T x}}{\sum_{j=1}^{K} e^{w_j^T x}}$$

In [ ]:
from sklearn.multiclass import OneVsRestClassifier

X3, y3 = make_classification(
    n_samples=600, n_features=2, n_redundant=0,
    n_classes=3, n_clusters_per_class=1, random_state=42
)
X3 = StandardScaler().fit_transform(X3)

x_min, x_max = X3[:,0].min()-1, X3[:,0].max()+1
y_min, y_max = X3[:,1].min()-1, X3[:,1].max()+1
xx3, yy3 = np.meshgrid(np.linspace(x_min, x_max, 300),
                        np.linspace(y_min, y_max, 300))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

models = [
    ('ovr', OneVsRestClassifier(LogisticRegression(solver='lbfgs', max_iter=1000))),
    ('multinomial', LogisticRegression(solver='lbfgs', max_iter=1000)),
]

for ax, (mc, model) in zip(axes, models):
    model.fit(X3, y3)

    Z = model.predict(np.c_[xx3.ravel(), yy3.ravel()]).reshape(xx3.shape)
    ax.contourf(xx3, yy3, Z, alpha=0.25, cmap='Set1')
    ax.scatter(X3[:,0], X3[:,1], c=y3, cmap='Set1',
               edgecolors='k', linewidths=0.4, s=30)
    ax.set_title(f'multi_class="{mc}"\nacc={model.score(X3, y3):.3f}', fontsize=11)
    ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
    ax.grid(alpha=0.3)

plt.suptitle('OvR vs Multinomial — границы решений (3 класса)', fontsize=13)
plt.tight_layout()
plt.show()


## 8. `fit_intercept` — свободный член

Определяет, добавлять ли в модель свободный член $w_0$ (сдвиг). Если `False` — граница решений будет проходить через начало координат.

- `True` (default) — модель подбирает $w_0$ самостоятельно
- `False` — $w_0 = 0$, граница решений вынуждена проходить через центр

Отключать стоит только если данные уже центрированы и вы уверены, что граница должна проходить через ноль.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

X_fi, y_fi = make_classification(
    n_samples=300, n_features=2, n_redundant=0,
    n_clusters_per_class=1, random_state=5
)
X_fi = StandardScaler().fit_transform(X_fi)

x_min, x_max = X_fi[:,0].min()-1, X_fi[:,0].max()+1
y_min, y_max = X_fi[:,1].min()-1, X_fi[:,1].max()+1
xx_fi, yy_fi = np.meshgrid(np.linspace(x_min, x_max, 300),
                             np.linspace(y_min, y_max, 300))

for ax, fi in zip(axes, [True, False]):
    model = LogisticRegression(fit_intercept=fi, max_iter=1000)
    model.fit(X_fi, y_fi)

    Z = model.predict(np.c_[xx_fi.ravel(), yy_fi.ravel()]).reshape(xx_fi.shape)
    ax.contourf(xx_fi, yy_fi, Z, alpha=0.2, cmap='RdBu')
    ax.scatter(X_fi[:,0], X_fi[:,1], c=y_fi, cmap='RdBu',
               edgecolors='k', linewidths=0.4, s=30)

    intercept = f"w0 = {model.intercept_[0]:.3f}" if fi else "w0 = 0"
    ax.set_title(f'fit_intercept={fi}\n{intercept},  acc={model.score(X_fi, y_fi):.3f}', fontsize=11)
    ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
    ax.grid(alpha=0.3)

plt.suptitle('Влияние fit_intercept на границу решений', fontsize=13)
plt.tight_layout()
plt.show()

## 9. `intercept_scaling` — масштаб свободного члена

Работает **только** при `solver='liblinear'` и `fit_intercept=True`.

При использовании `liblinear` к каждому объекту добавляется искусственный признак со значением `intercept_scaling`. Это влияет на то, насколько сильно регуляризация затрагивает свободный член.

- Большое значение → свободный член меньше подвержен регуляризации
- По умолчанию: `intercept_scaling=1.0`

На практике меняется редко.

In [ ]:
scaling_values = [0.01, 0.1, 1.0, 10.0, 100.0]

intercepts = []
for s in scaling_values:
    model = LogisticRegression(solver='liblinear', intercept_scaling=s, max_iter=1000)
    model.fit(X_scaled, y)
    intercepts.append(model.intercept_[0])

plt.figure(figsize=(7, 4))
plt.plot([str(s) for s in scaling_values], intercepts, 'o-', color='steelblue', linewidth=2)
plt.xlabel('intercept_scaling')
plt.ylabel('Значение intercept (w0)')
plt.title('Влияние intercept_scaling на свободный член', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. `l1_ratio` — баланс между L1 и L2 в ElasticNet

Используется только при `penalty='elasticnet'`. Определяет, насколько штраф смещён в сторону L1 или L2:

$$\text{penalty} = \text{l1\_ratio} \cdot \|w\|_1 + (1 - \text{l1\_ratio}) \cdot \frac{1}{2}\|w\|_2^2$$

- `l1_ratio=0` — чистый L2 (как Ridge)
- `l1_ratio=1` — чистый L1 (как Lasso)
- `l1_ratio=0.5` (default) — равный баланс

По умолчанию: `l1_ratio=None`, обязательно задавать при `penalty='elasticnet'`.

In [ ]:
l1_ratios = [0.0, 0.25, 0.5, 0.75, 1.0]
results = []

for r in l1_ratios:
    model = LogisticRegression(penalty='elasticnet', solver='saga',
                                l1_ratio=r, C=0.1, max_iter=2000)
    model.fit(X_scaled, y)
    coefs = model.coef_[0]
    results.append(coefs)

fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)

for ax, coefs, r in zip(axes, results, l1_ratios):
    n_zero = np.sum(np.abs(coefs) < 1e-4)
    colors = ['tomato' if abs(c) < 1e-4 else 'steelblue' for c in coefs]
    ax.bar(range(len(coefs)), coefs, color=colors, alpha=0.8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'l1_ratio={r}\nнулевых: {n_zero}/30', fontsize=10)
    ax.set_xlabel('Признак')
    ax.grid(alpha=0.3)

axes[0].set_ylabel('Значение коэффициента')
plt.suptitle('Влияние l1_ratio на коэффициенты (penalty=elasticnet, C=0.1)', fontsize=13)
plt.tight_layout()
plt.show()

## 11. `warm_start` — продолжение обучения

При `warm_start=True` модель не начинает обучение с нуля, а продолжает с весов предыдущего вызова `fit()`.

- `False` (default) — каждый `fit()` начинает заново
- `True` — каждый `fit()` продолжает с предыдущего состояния

Полезно когда данные поступают порциями или нужно дообучить модель без полного перезапуска.

In [ ]:
iters_cold, iters_warm = [], []
batch_sizes = [50, 100, 200, 300, 400, 500]

model_cold = LogisticRegression(solver='saga', max_iter=1000, warm_start=False)
for n in batch_sizes:
    model_cold.fit(X_scaled[:n], y[:n])
    iters_cold.append(model_cold.n_iter_[0])

model_warm = LogisticRegression(solver='saga', max_iter=1000, warm_start=True)
for n in batch_sizes:
    model_warm.fit(X_scaled[:n], y[:n])
    iters_warm.append(model_warm.n_iter_[0])

plt.figure(figsize=(8, 4))
plt.plot(batch_sizes, iters_cold, 'o-', color='tomato', linewidth=2, label='warm_start=False')
plt.plot(batch_sizes, iters_warm, 's-', color='steelblue', linewidth=2, label='warm_start=True')
plt.xlabel('Размер обучающей выборки')
plt.ylabel('Итераций до сходимости')
plt.title('warm_start: итерации при дообучении на новых данных', fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 12. `random_state` — воспроизводимость

Фиксирует случайность в алгоритмах `sag` и `saga`, которые используют случайную выборку данных на каждой итерации.

- `None` (default) — каждый запуск может дать немного разный результат
- `int` — фиксирует результат, запуск всегда одинаковый

На результат практически не влияет, важен для **воспроизводимости экспериментов**.

## 13. `n_jobs` — параллельное вычисление

Задаёт количество потоков при решении многоклассовой задачи (`multi_class='ovr'`).

- `None` (default) — один поток
- `-1` — использовать все доступные ядра процессора
- `int` — конкретное число потоков

На результат не влияет, только на скорость. Эффект заметен только при большом числе классов.

## 14. `verbose` — логирование процесса обучения

Управляет выводом информации в процессе обучения.

- `0` (default) — ничего не выводит
- `1` и выше — выводит прогресс итераций

Полезен для отладки при больших датасетах, чтобы видеть что модель вообще обучается.

# Часть 2: Продвинутые инструменты для логистической регрессии

В этой части разбираем:
1. `LogisticRegressionCV` — версия с автоматическим подбором настроек
2. `RidgeClassifier` — быстрый классификатор с защитой от переобучения
3. `Lasso` — инструмент для задач регрессии с автоотбором признаков
4. Сравнение всех моделей
5. Шпаргалка: что и когда выбирать


In [ ]:
# Всё что понадобится для этой части
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings
import time

from sklearn.linear_model import (
    LogisticRegression,
    LogisticRegressionCV,
    RidgeClassifier,
    Lasso,
    Ridge
)
from sklearn.datasets import load_breast_cancer, make_regression, make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Загружаем данные — тот же датасет, что использует напарник
data = load_breast_cancer()
X, y = data.data, data.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"Данные загружены. Объектов: {X.shape[0]}, признаков: {X.shape[1]}")
print(f"Обучающая выборка: {X_train.shape[0]}, тестовая: {X_test.shape[0]}")


---
## 1. LogisticRegressionCV — модель, которая сама подбирает настройки

### Проблема

В параметре `C` (сила регуляризации) мы уже разобрались в предыдущей части. Но как понять, какое именно значение `C` поставить — 0.01, 1.0 или 100?

Обычно это делается вручную: перебираешь разные значения, смотришь где лучше. Это долго и неудобно.

### Решение — `LogisticRegressionCV`

Это та же логистическая регрессия, но со встроенным автоматическим подбором `C`. Вы просто говорите ей «попробуй 20 вариантов» — и она сама находит лучший.

**Как это работает внутри:**
1. Данные делятся на несколько частей (например, 5)
2. Модель обучается на 4 частях, проверяется на 1-й
3. Так повторяется для каждой части
4. Для каждого значения `C` считается средняя точность
5. Побеждает то `C`, при котором точность была выше всего

Это и называется **кросс-валидация** — честная проверка, потому что модель всегда проверяется на данных, которых не видела при обучении.

### Параметры

| Параметр | Простыми словами | По умолчанию |
|---|---|---|
| `Cs` | Сколько вариантов `C` попробовать (число или список) | `10` |
| `cv` | На сколько частей делить данные для проверки | `5` |
| `scoring` | По какому критерию выбирать лучшее `C` | `'accuracy'` |
| `penalty` | Тип защиты от переобучения | `'l2'` |
| `solver` | Алгоритм вычислений (как в обычной LogisticRegression) | `'lbfgs'` |
| `refit` | После выбора лучшего C — переобучить на всех данных? | `True` |
| `max_iter` | Максимум шагов при обучении | `100` |

После обучения появляются два полезных атрибута:
- `C_` — найденное лучшее значение C
- `scores_` — таблица с результатами для каждого C и каждой части данных


In [ ]:
# Обучаем LogisticRegressionCV
lrcv = LogisticRegressionCV(
    Cs=20,              # попробовать 20 разных значений C
    cv=5,               # делить данные на 5 частей
    scoring='accuracy', # выбирать по точности
    max_iter=2000,
    random_state=42
)
lrcv.fit(X_train, y_train)

print(f"Лучшее C, которое нашла модель: {lrcv.C_[0]:.6f}")
print(f"Точность на тестовых данных:    {lrcv.score(X_test, y_test):.4f}")


In [ ]:
# Смотрим как менялась точность при разных C
# scores_ хранит результаты всех проверок
scores_array = list(lrcv.scores_.values())[0]  # форма: (5 частей, 20 значений C)
mean_scores = scores_array.mean(axis=0)         # среднее по всем частям
std_scores  = scores_array.std(axis=0)          # разброс

fig, ax = plt.subplots(figsize=(9, 4))

ax.semilogx(lrcv.Cs_, mean_scores, 'o-', color='steelblue', linewidth=2, label='Средняя точность')
ax.fill_between(lrcv.Cs_,
                mean_scores - std_scores,
                mean_scores + std_scores,
                alpha=0.2, color='steelblue', label='Разброс ± std')
ax.axvline(lrcv.C_[0], color='tomato', linestyle='--', linewidth=2,
           label=f'Лучшее C = {lrcv.C_[0]:.4f}')

ax.set_xlabel('Значение C (ось логарифмическая)')
ax.set_ylabel('Точность (accuracy)')
ax.set_title('LogisticRegressionCV: как менялась точность при разных C', fontsize=12)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Красная пунктирная линия — это лучшее C, которое модель выбрала сама.")


### Итог по LogisticRegressionCV

`LogisticRegressionCV` = `LogisticRegression` + автоматический подбор `C`.

Используйте её когда не знаете, какое `C` поставить. Она сделает перебор за вас и выберет лучшее.


---
## 2. RidgeClassifier — быстрый классификатор с L2-защитой

### Что это такое

`RidgeClassifier` — это ещё один способ решать задачи классификации (то есть когда нужно отнести объект к одному из классов).

Он работает иначе, чем логистическая регрессия:
- Логистическая регрессия: учится напрямую определять классы, умеет говорить «вероятность 87%»
- RidgeClassifier: сначала предсказывает число (как в задаче регрессии), потом по знаку числа определяет класс: положительное → класс 1, отрицательное → класс 0

**Когда выбирать RidgeClassifier:**
- Когда важна скорость (он обучается быстрее)
- Когда вероятности классов не нужны — нужен только ответ «да/нет»

**Когда НЕ выбирать:**
- Если нужен метод `predict_proba()` (вероятности) — его у RidgeClassifier нет

### Сравнение с LogisticRegression

| | LogisticRegression (L2) | RidgeClassifier |
|---|---|---|
| Вероятности `predict_proba` | ✅ Есть | ❌ Нет |
| Скорость | Обычная | Быстрее |
| Параметр силы защиты | `C` (чем меньше — тем сильнее) | `alpha` (чем больше — тем сильнее) |

⚠️ Обратите внимание: логика параметра **обратная**!
- В `LogisticRegression`: маленький `C` = сильная защита
- В `RidgeClassifier`: большой `alpha` = сильная защита

### Параметры

| Параметр | Простыми словами | По умолчанию |
|---|---|---|
| `alpha` | Сила L2-защиты от переобучения | `1.0` |
| `fit_intercept` | Добавлять ли сдвиг (свободный член) | `True` |
| `class_weight` | Веса для классов при несбалансированных данных | `None` |
| `solver` | Метод вычислений | `'auto'` |


In [ ]:
# Сравниваем скорость и точность RidgeClassifier vs LogisticRegression
сравнение = [
    ('LogisticRegression (L2)',     LogisticRegression(C=1.0, max_iter=2000)),
    ('RidgeClassifier (alpha=0.1)', RidgeClassifier(alpha=0.1)),
    ('RidgeClassifier (alpha=1)',   RidgeClassifier(alpha=1.0)),
    ('RidgeClassifier (alpha=10)',  RidgeClassifier(alpha=10.0)),
]

print(f"{'Модель':<35} {'Точность':>10} {'Время (мс)':>12}")
print("-" * 60)
for name, model in сравнение:
    t0 = time.time()
    model.fit(X_train, y_train)
    elapsed = (time.time() - t0) * 1000
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"{name:<35} {acc:>10.4f} {elapsed:>11.2f}")


In [ ]:
# Как параметр alpha влияет на модель
alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
нормы_весов = []
точности = []

for alpha in alphas:
    model = RidgeClassifier(alpha=alpha)
    model.fit(X_train, y_train)
    нормы_весов.append(np.linalg.norm(model.coef_))
    точности.append(accuracy_score(y_test, model.predict(X_test)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].semilogx(alphas, нормы_весов, 'o-', color='steelblue', linewidth=2)
axes[0].set_xlabel('alpha')
axes[0].set_ylabel('Размер весов модели')
axes[0].set_title('Чем больше alpha — тем меньше веса', fontsize=11)
axes[0].grid(alpha=0.3)

axes[1].semilogx(alphas, точности, 's-', color='tomato', linewidth=2)
axes[1].set_xlabel('alpha')
axes[1].set_ylabel('Точность на тестовых данных')
axes[1].set_title('Как alpha влияет на точность', fontsize=11)
axes[1].set_ylim(min(точности) - 0.01, 1.0)
axes[1].grid(alpha=0.3)

plt.suptitle('RidgeClassifier: влияние параметра alpha', fontsize=13)
plt.tight_layout()
plt.show()


### Итог по RidgeClassifier

`RidgeClassifier` — быстрая альтернатива логистической регрессии. Используйте когда нужна скорость и не нужны вероятности. Главный параметр — `alpha`: чем он больше, тем сильнее защита от переобучения.


---
## 3. Lasso — регрессия с автоматическим отбором признаков

### Сначала важное уточнение

`Lasso` — это модель для задач **регрессии**, то есть когда нужно предсказать число (цену квартиры, температуру и т.д.), а не класс.

Зачем она здесь? Потому что в `LogisticRegression` есть параметр `penalty='l1'`, и именно оттуда пришла идея L1-защиты. Чтобы понять как L1 работает — лучше всего посмотреть на `Lasso`.

### Главная особенность Lasso

Когда у вас много признаков (столбцов в таблице), не все из них реально влияют на результат. Часть из них — просто шум.

- **Ridge (L2)**: уменьшает веса всех признаков, но не убирает их полностью
- **Lasso (L1)**: обнуляет веса неважных признаков — они перестают влиять на результат

Это называется **автоматический отбор признаков**: Lasso сам решает, какие признаки оставить, а какие выбросить.

### Параметры

| Параметр | Простыми словами | По умолчанию |
|---|---|---|
| `alpha` | Сила L1-защиты. Чем больше — тем больше признаков обнулится | `1.0` |
| `fit_intercept` | Добавлять сдвиг | `True` |
| `max_iter` | Максимум шагов при обучении | `1000` |
| `tol` | Насколько точно считать — меньше значение, точнее но медленнее | `1e-4` |
| `warm_start` | Продолжать обучение с прошлого раза (а не начинать заново) | `False` |
| `positive` | Запретить отрицательные коэффициенты | `False` |

### Связь с LogisticRegression

| | Lasso | LogisticRegression(penalty='l1') |
|---|---|---|
| Задача | Регрессия (предсказание числа) | Классификация (предсказание класса) |
| Тип защиты | L1 | L1 |
| Эффект | Обнуляет признаки | Обнуляет признаки |

Одна и та же идея L1 — применяется в разных задачах.


In [ ]:
# Создаём задачу регрессии специально с "лишними" признаками
# 20 признаков, но только 5 из них реально влияют на результат
X_reg, y_reg = make_regression(
    n_samples=200,
    n_features=20,
    n_informative=5,  # только 5 из 20 реально важны
    noise=10,
    random_state=42
)
X_reg = StandardScaler().fit_transform(X_reg)

lasso = Lasso(alpha=1.0, max_iter=5000)
ridge = Ridge(alpha=1.0)

lasso.fit(X_reg, y_reg)
ridge.fit(X_reg, y_reg)

n_zero_lasso = np.sum(np.abs(lasso.coef_) < 1e-4)
n_zero_ridge = np.sum(np.abs(ridge.coef_) < 1e-4)

print(f"Lasso: обнулил {n_zero_lasso} признаков из {X_reg.shape[1]}")
print(f"Ridge: обнулил {n_zero_ridge} признаков из {X_reg.shape[1]}")
print()
print("Lasso нашёл почти точно: реально важных признаков было 5")


In [ ]:
# Сравниваем Lasso и Ridge визуально
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Lasso
цвета_lasso = ['tomato' if abs(c) < 1e-4 else 'steelblue' for c in lasso.coef_]
axes[0].bar(range(20), lasso.coef_, color=цвета_lasso, alpha=0.85)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title(f'Lasso: обнулил {n_zero_lasso} из 20 признаков', fontsize=11)
axes[0].set_xlabel('Номер признака')
axes[0].set_ylabel('Вес признака')
axes[0].grid(alpha=0.3)
axes[0].legend(handles=[
    Patch(color='steelblue', label='Оставил'),
    Patch(color='tomato', label='Обнулил')
])

# Ridge
axes[1].bar(range(20), ridge.coef_, color='steelblue', alpha=0.85)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Ridge: уменьшил все признаки, но не обнулил', fontsize=11)
axes[1].set_xlabel('Номер признака')
axes[1].set_ylabel('Вес признака')
axes[1].grid(alpha=0.3)

plt.suptitle('Lasso убирает лишние признаки, Ridge — просто уменьшает все', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Как alpha в Lasso влияет на число оставшихся признаков
alphas_lasso = np.logspace(-3, 2, 50)
оставшиеся = []

for a in alphas_lasso:
    m = Lasso(alpha=a, max_iter=10000)
    m.fit(X_reg, y_reg)
    оставшиеся.append(np.sum(np.abs(m.coef_) >= 1e-4))

plt.figure(figsize=(8, 4))
plt.semilogx(alphas_lasso, оставшиеся, 'o-', color='steelblue', linewidth=2, markersize=4)
plt.axhline(5, color='tomato', linestyle='--', linewidth=2,
            label='Правильный ответ: 5 важных признаков')
plt.xlabel('alpha (чем правее — тем сильнее защита)')
plt.ylabel('Сколько признаков оставила модель')
plt.title('Lasso: чем больше alpha — тем меньше признаков остаётся', fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("При правильном alpha модель сама находит нужные 5 признаков!")


### Итог по Lasso

`Lasso` — это регрессия с L1-защитой. Её главная фишка: она сама выбрасывает ненужные признаки, обнуляя их веса. Чем больше `alpha` — тем больше признаков она выбросит.

В классификации то же самое делает `LogisticRegression(penalty='l1')`.


---
## 4. Сравнение всех моделей

Теперь запустим все модели на одном датасете и сравним результаты.


In [ ]:
# Все модели на одном датасете
все_модели = {
    'LogReg (L2)':          LogisticRegression(C=1.0, max_iter=2000),
    'LogReg (L1)':          LogisticRegression(penalty='l1', solver='saga', C=1.0, max_iter=2000),
    'LogReg (ElasticNet)':  LogisticRegression(penalty='elasticnet', solver='saga',
                                               l1_ratio=0.5, C=1.0, max_iter=2000),
    'LogisticRegressionCV': LogisticRegressionCV(Cs=10, cv=5, max_iter=2000),
    'RidgeClassifier':      RidgeClassifier(alpha=1.0),
}

строки = []
for название, модель in все_модели.items():
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        t0 = time.time()
        cv = cross_val_score(модель, X_scaled, y, cv=5, scoring='accuracy')
        модель.fit(X_train, y_train)
        прошло = time.time() - t0
        прогноз = модель.predict(X_test)
        есть_вероятности = hasattr(модель, 'predict_proba')
        auc = roc_auc_score(y_test, модель.predict_proba(X_test)[:,1]) if есть_вероятности else None
        строки.append({
            'Модель':           название,
            'CV Accuracy':      f"{cv.mean():.4f} ± {cv.std():.4f}",
            'Тест Accuracy':    f"{accuracy_score(y_test, прогноз):.4f}",
            'F1':               f"{f1_score(y_test, прогноз):.4f}",
            'ROC-AUC':          f"{auc:.4f}" if auc else "нет",
            'Время (мс)':       f"{прошло*1000:.1f}",
        })

df = pd.DataFrame(строки).set_index('Модель')
print(df.to_string())


In [ ]:
# График: сравнение точности
названия = []
средние = []
разбросы = []

for название, модель in все_модели.items():
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        cv = cross_val_score(модель, X_scaled, y, cv=5, scoring='accuracy')
    названия.append(название)
    средние.append(cv.mean())
    разбросы.append(cv.std())

цвета = ['steelblue', 'tomato', 'mediumseagreen', 'mediumpurple', 'orange']

fig, ax = plt.subplots(figsize=(10, 5))
бары = ax.barh(названия, средние, xerr=разбросы, color=цвета, alpha=0.85,
               error_kw=dict(capsize=5, ecolor='black', linewidth=1.5))

for бар, среднее in zip(бары, средние):
    ax.text(среднее + 0.001, бар.get_y() + бар.get_height()/2,
            f'{среднее:.4f}', va='center', fontsize=10)

ax.set_xlim(min(средние) - 0.02, 1.01)
ax.set_xlabel('Точность (5-fold кросс-валидация)')
ax.set_title('Сравнение моделей по точности', fontsize=13)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# График: кто убирает признаки, а кто — нет
модели_признаки = {
    'LogReg L2 (C=1)':      LogisticRegression(C=1.0, max_iter=2000),
    'LogReg L1 (C=1)':      LogisticRegression(penalty='l1', solver='saga', C=1.0, max_iter=2000),
    'LogReg L1 (C=0.1)':    LogisticRegression(penalty='l1', solver='saga', C=0.1, max_iter=2000),
    'LogReg ElasticNet':    LogisticRegression(penalty='elasticnet', solver='saga',
                                               l1_ratio=0.5, C=0.5, max_iter=2000),
    'RidgeClassifier':      RidgeClassifier(alpha=1.0),
}

кол_признаков = []
for название, модель in модели_признаки.items():
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        модель.fit(X_train, y_train)
    кол_признаков.append(np.sum(np.abs(модель.coef_[0]) >= 1e-4))

цвета_признаки = ['tomato' if n < X_train.shape[1] else 'steelblue' for n in кол_признаков]

fig, ax = plt.subplots(figsize=(9, 4))
бары = ax.barh(list(модели_признаки.keys()), кол_признаков, color=цвета_признаки, alpha=0.85)
ax.axvline(X_train.shape[1], color='gray', linestyle='--', alpha=0.7,
           label=f'Всего признаков = {X_train.shape[1]}')
for бар, n in zip(бары, кол_признаков):
    ax.text(n + 0.3, бар.get_y() + бар.get_height()/2, str(n), va='center', fontsize=10)

ax.set_xlabel('Число активных признаков')
ax.set_title('Синие — используют все признаки, красные — часть убрали', fontsize=12)
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()


---
## 5. Шпаргалка: что выбирать и когда

### Выбор модели

```
Задача классификации (нужно предсказать класс)?
│
├── Нужны вероятности (насколько уверена модель)?
│   └── Да → LogisticRegression
│
├── Не знаю какое C ставить
│   └── → LogisticRegressionCV — сама найдёт лучшее
│
├── Важна скорость, вероятности не нужны
│   └── → RidgeClassifier
│
└── Хочу убрать лишние признаки автоматически
    └── → LogisticRegression(penalty='l1')

Задача регрессии (нужно предсказать число)?
│
├── Хочу убрать лишние признаки → Lasso
└── Просто регуляризация       → Ridge
```

### Выбор параметра C / alpha

| Симптом | Что делать |
|---|---|
| Модель переобучается (на обучающих данных хорошо, на тестовых — плохо) | Уменьшить `C` или увеличить `alpha` |
| Модель недообучается (везде плохо) | Увеличить `C` или уменьшить `alpha` |
| Не знаю что делать | Использовать `LogisticRegressionCV` |

### Выбор penalty

| Ситуация | Что ставить |
|---|---|
| Стандартная задача | `l2` (по умолчанию) |
| Много лишних признаков, хочу их убрать | `l1` + `solver='saga'` |
| Хочу и то и другое | `elasticnet` + `solver='saga'` + `l1_ratio=0.5` |


In [ ]:
# Итоговая таблица-шпаргалка
шпаргалка = pd.DataFrame([
    ['Стандартная классификация',           'LogisticRegression',    'l2',      'lbfgs', 'C=1.0'],
    ['Не знаю какое C',                     'LogisticRegressionCV',  'l2',      'lbfgs', 'Cs=20, cv=5'],
    ['Нужна скорость, без вероятностей',    'RidgeClassifier',       'L2',      '—',     'alpha=1.0'],
    ['Много лишних признаков',              'LogisticRegression',    'l1',      'saga',  'C=0.1'],
    ['Несбалансированные классы',           'LogisticRegression',    'l2',      'lbfgs', "class_weight='balanced'"],
    ['Регрессия + убрать лишние признаки',  'Lasso',                 'L1',      '—',     'alpha=1.0'],
], columns=['Ситуация', 'Модель', 'Тип защиты', 'solver', 'Ключевой параметр'])

print(шпаргалка.to_string(index=False))


## Вопросы для самопроверки

**1. Чем отличается penalty='l1' от penalty='l2'?**

`penalty='l2'` уменьшает веса всех признаков, но обычно не делает их ровно нулевыми. Это помогает сгладить модель и бороться с переобучением.  
`penalty='l1'` может занулять часть коэффициентов, поэтому работает как автоматический отбор признаков: неважные признаки перестают влиять на предсказание.

**2. У вас несбалансированный датасет: 95% класс 0, 5% класс 1. Модель предсказывает всегда класс 0 и получает accuracy=0.95. Какой параметр поможет исправить ситуацию и как?**

Поможет `class_weight='balanced'` или ручной словарь весов, например `{0: 1, 1: 10}`. Этот параметр увеличивает штраф за ошибки на редком классе, поэтому модель перестаёт игнорировать класс 1 и чаще пытается его находить. Accuracy может немного снизиться, но recall редкого класса обычно растёт.

**3. При обучении модели появился ConvergenceWarning. Назовите два способа исправить это.**

Первый способ — увеличить `max_iter`, чтобы дать оптимизатору больше шагов для сходимости.  
Второй способ — масштабировать признаки через `StandardScaler`, потому что логистическая регрессия чувствительна к разным масштабам. Также можно выбрать другой `solver` или ослабить слишком сильную регуляризацию.

**4. Вы хотите использовать penalty='l1', но получаете ошибку при solver='lbfgs'. Почему и как исправить?**

`lbfgs` не поддерживает L1-регуляризацию. Нужно заменить solver на совместимый, например `solver='liblinear'` для L1/L2 на небольших данных или `solver='saga'`, если нужны L1, ElasticNet или большие данные.

**5. В чём разница между уменьшением C и увеличением max_iter — на что влияет каждый параметр?**

`C` управляет силой регуляризации: чем меньше `C`, тем сильнее штраф за большие веса и тем проще модель.  
`max_iter` управляет только лимитом итераций оптимизации: увеличение `max_iter` помогает модели досчитаться до сходимости, но само по себе не делает регуляризацию сильнее или слабее.


---
## Вопросы для самопроверки

**1. Чем `LogisticRegressionCV` отличается от обычного `LogisticRegression`? Что она делает автоматически?**

**2. У `RidgeClassifier` нет метода `predict_proba()`. Почему? Как он вообще определяет класс?**

**3. В `RidgeClassifier` параметр `alpha`, а в `LogisticRegression` — параметр `C`. Как они соотносятся? Если хочу усилить защиту от переобучения — мне нужно увеличить или уменьшить каждый из них?**

**4. У вас таблица с 50 столбцами, но вы подозреваете, что реально важных — только 5–10. Какую модель и с каким `penalty` выбрать?**

**5. `Lasso` — это для классификации или регрессии? Что общего между `Lasso` и `LogisticRegression(penalty='l1')`?**

**6. Вы обучили `LogisticRegressionCV` с `Cs=20, cv=5`. Как узнать, какое значение `C` она выбрала? Какой атрибут модели за это отвечает?**
